# BACKTESTING_272 — Téléchargement des données
 
**Univers** : STOXX Europe 600  
**Période téléchargment** : 2015-09-01 --> 2026-02-20 --> pour téléchargement de données et avoir de l'historique pour le calcul des signaux
**Benchmark** : ESTRON Index
**Backtest periode** : 2016-12-31 --> 2026-02-20


## 1. Importation des Libraries <a id='1'></a>

In [ ]:
import sys
print(sys.executable)

import pyarrow
print(pyarrow.__version__)

from pathlib import Path

project_root = Path().resolve().parent          # BACKTESTING_M2272
sys.path.insert(0, str(project_root / "src"))   # ajoute src au PYTHONPATH

from request.request_unique import BloombergHistoricalData
from request.utils import ParquetStockage, ExportData, MonthEndExtractor, BloombergDateFormatter, IndexMembersParquetMerger
from request.request_index_members import IndexMembersDownloader


## 2. Requete historique de l'Indice Univers <a id='1'></a>

**Objectif** : Déterminer les dates de requetes fin de mois (date de rebalancement) des composants du stoxx 600 Europe

### 2a. Requete Benchmark SXXP Index

In [ ]:
# Requete
bloom = BloombergHistoricalData()

df = bloom.get_history(ticker="SXXP Index", field="PX_LAST", start_date="20160901", end_date="20260220", frequency="DAILY")

# Sauvegarde
stockage = ParquetStockage()

stockage.sauvegarder(df, nom_fichier="SXXP Index_PX_LAST",ticker="SXXP Index")

"""
- chemin de stockage : /Users/arthurlenet/Desktop/Backtesting_M2272/data/initial_data
"""

In [ ]:
# Visualisation du fichiers .parquet stocké
stockage = ParquetStockage()

sxxp_index_px_last = stockage.charger("SXXP Index_PX_LAST")

sxxp_index_px_last.tail()

,Date,SXXP Index
2427,2026-02-16,618.52
2428,2026-02-17,621.29
2429,2026-02-18,628.69
2430,2026-02-19,625.33
2431,2026-02-20,630.56


### 2b. Requete Risk free Rate/ Benchmark :  ESTRON Index €STER

In [ ]:
# Requete
bloom = BloombergHistoricalData()

df = bloom.get_history(ticker="ESTRON Index", field="PX_LAST", start_date="20160901", end_date="20260220", frequency="DAILY")

# Sauvegarde
stockage = ParquetStockage()

stockage.sauvegarder(df, nom_fichier="ESTRON Index_PX_LAST", ticker="ESTRON Index")

In [ ]:
# Visualisation du fichier téléchargé
estron_index_px_last = stockage.charger("ESTRON Index_PX_LAST")
estron_index_px_last.tail()

,Date,ESTRON Index
1633,2026-02-16,1.929
1634,2026-02-17,1.931
1635,2026-02-18,1.931
1636,2026-02-19,1.933
1637,2026-02-20,1.932


Verification des données

In [ ]:
# Export
exporter = ExportData(folder_type="verification")
exporter.export_df(sxxp_index_px_last, "sxxp_index_px_last")
exporter.export_df(estron_index_px_last, "estron_index_px_last")

WindowsPath('C:/Users/lenar25/Documents/Backtesting_M2272/data/verification/estron_index_px_last_20260226_173300.xlsx')


## 3. Requete historique des composants de l'Indice Univers <a id='1'></a>

**Objectif** : Déterminer les composants du stoxx 600 Europe à chaque date de rebalancement pour éviter les biais de survivants

### 3a. Identifier les dates de fin de chaque mois

In [ ]:
# Load des données
stockage = ParquetStockage()

sxxp_index_px_last = stockage.charger("SXXP Index_PX_LAST")

sxxp_index_px_last.tail()

,Date,SXXP Index
2427,2026-02-16,618.52
2428,2026-02-17,621.29
2429,2026-02-18,628.69
2430,2026-02-19,625.33
2431,2026-02-20,630.56


In [ ]:
# extraction des dates de fin de mois
extractor = MonthEndExtractor(sxxp_index_px_last)
month_end_dates = extractor.get_month_end_dates()

# Exporter pour stockage
exporter = ExportData(folder_type="initial_data")
exporter.export_df(month_end_dates, "month_end_dates")

# mettre en liste les dates de fin de mois
month_end_dates_list = month_end_dates.to_list()

In [ ]:
# visualisation
month_end_dates_list

[Timestamp('2016-09-30 00:00:00'),
 Timestamp('2016-10-31 00:00:00'),
 Timestamp('2016-11-30 00:00:00'),
 Timestamp('2016-12-30 00:00:00'),
 Timestamp('2017-01-31 00:00:00'),
 Timestamp('2017-02-28 00:00:00'),
 Timestamp('2017-03-31 00:00:00'),
 Timestamp('2017-04-28 00:00:00'),
 Timestamp('2017-05-31 00:00:00'),
 Timestamp('2017-06-30 00:00:00'),
 Timestamp('2017-07-31 00:00:00'),
 Timestamp('2017-08-31 00:00:00'),
 Timestamp('2017-09-29 00:00:00'),
 Timestamp('2017-10-31 00:00:00'),
 Timestamp('2017-11-30 00:00:00'),
 Timestamp('2017-12-29 00:00:00'),
 Timestamp('2018-01-31 00:00:00'),
 Timestamp('2018-02-28 00:00:00'),
 Timestamp('2018-03-29 00:00:00'),
 Timestamp('2018-04-30 00:00:00'),
 Timestamp('2018-05-31 00:00:00'),
 Timestamp('2018-06-29 00:00:00'),
 Timestamp('2018-07-31 00:00:00'),
 Timestamp('2018-08-31 00:00:00'),
 Timestamp('2018-09-28 00:00:00'),
 Timestamp('2018-10-31 00:00:00'),
 Timestamp('2018-11-30 00:00:00'),
 Timestamp('2018-12-31 00:00:00'),
 Timestamp('2019-01-

In [ ]:
# Reformatter les dates dans le format Bloomberg pour le téléchargments bloomberg des composants
formatter = BloombergDateFormatter(month_end_dates_list)

bloomberg_month_end_dates_list = formatter.to_yyyymmdd()

bloomberg_month_end_dates_list

[20160930,
 20161031,
 20161130,
 20161230,
 20170131,
 20170228,
 20170331,
 20170428,
 20170531,
 20170630,
 20170731,
 20170831,
 20170929,
 20171031,
 20171130,
 20171229,
 20180131,
 20180228,
 20180329,
 20180430,
 20180531,
 20180629,
 20180731,
 20180831,
 20180928,
 20181031,
 20181130,
 20181231,
 20190131,
 20190228,
 20190329,
 20190430,
 20190531,
 20190628,
 20190731,
 20190830,
 20190930,
 20191031,
 20191129,
 20191231,
 20200131,
 20200228,
 20200331,
 20200430,
 20200529,
 20200630,
 20200731,
 20200831,
 20200930,
 20201030,
 20201130,
 20201231,
 20210129,
 20210226,
 20210331,
 20210430,
 20210531,
 20210630,
 20210730,
 20210831,
 20210930,
 20211029,
 20211130,
 20211231,
 20220131,
 20220228,
 20220331,
 20220429,
 20220531,
 20220630,
 20220729,
 20220831,
 20220930,
 20221031,
 20221130,
 20221230,
 20230131,
 20230228,
 20230331,
 20230428,
 20230531,
 20230630,
 20230731,
 20230831,
 20230929,
 20231031,
 20231130,
 20231229,
 20240131,
 20240229,
 20240328,

### 3b. Téléchargement des composants historiques de l'Indice Univers à chaque date de fin de mois

In [ ]:
downloader = IndexMembersDownloader()

downloader.download_members(index_ticker="SXXP Index",dates=bloomberg_month_end_dates_list)

Sauvegardé : C:\Users\lenar25\Documents\Backtesting_M2272\data\initial_data\index_memb_20160930.parquet
Sauvegardé : C:\Users\lenar25\Documents\Backtesting_M2272\data\initial_data\index_memb_20161031.parquet
Sauvegardé : C:\Users\lenar25\Documents\Backtesting_M2272\data\initial_data\index_memb_20161130.parquet
Sauvegardé : C:\Users\lenar25\Documents\Backtesting_M2272\data\initial_data\index_memb_20161230.parquet
Sauvegardé : C:\Users\lenar25\Documents\Backtesting_M2272\data\initial_data\index_memb_20170131.parquet
Sauvegardé : C:\Users\lenar25\Documents\Backtesting_M2272\data\initial_data\index_memb_20170228.parquet
Sauvegardé : C:\Users\lenar25\Documents\Backtesting_M2272\data\initial_data\index_memb_20170331.parquet
Sauvegardé : C:\Users\lenar25\Documents\Backtesting_M2272\data\initial_data\index_memb_20170428.parquet
Sauvegardé : C:\Users\lenar25\Documents\Backtesting_M2272\data\initial_data\index_memb_20170531.parquet
Sauvegardé : C:\Users\lenar25\Documents\Backtesting_M2272\data\i

In [ ]:
# Test Visualisation
stockage = ParquetStockage()

index_memb_20260220 = stockage.charger("index_memb_20260220")

index_memb_20260220.tail()

,Date,Ticker
595,1970-01-01,YAR NO
596,1970-01-01,ZAL GY
597,1970-01-01,ZEAL DC
598,1970-01-01,ZEG LN
599,1970-01-01,ZURN SE


### 3c. Rassembler l'ensemble des composants historiques dans un seul fichier

In [17]:
merger = IndexMembersParquetMerger()
path_saved = merger.merge_to_parquet()

[INFO] Project root détecté : C:\Users\lenar25\Documents\Backtesting_M2272
[INFO] Dossier cible        : C:\Users\lenar25\Documents\Backtesting_M2272\data\initial_data
[INFO] Sortie attendue : C:\Users\lenar25\Documents\Backtesting_M2272\data\initial_data\historical_index_memb.parquet
[INFO] 114 fichiers fusionnés
[INFO] Shape final : (68402, 2)
[SUCCESS] Fichier créé : C:\Users\lenar25\Documents\Backtesting_M2272\data\initial_data\historical_index_memb.parquet (0.03 MB)


### 3d. Identifier les tickers des composants historiques

In [ ]:
# Load des données
stockage = ParquetStockage()

historical_index_memb = stockage.charger("historical_index_memb")

historical_index_memb.tail()

,Date,Ticker
68397,1970-01-01,YAR NO
68398,1970-01-01,ZAL GY
68399,1970-01-01,ZEAL DC
68400,1970-01-01,ZEG LN
68401,1970-01-01,ZURN SE


In [ ]:
# Anlayse
# taille de la liste
len(historical_index_memb["Ticker"])

# mettre en liste les tickers uniques historique de l'indice stoxx 600 Europe
ticker_list_historical_index_memb_unique = historical_index_memb["Ticker"].unique().tolist()

len(ticker_list_historical_index_memb_unique)

980

In [6]:
ticker_list_historical_index_memb_unique

['1844030D NA',
 '1986986D LN',
 '1CNHI IM',
 '1COV GY',
 '2462936D LN',
 '2578464D GY',
 '2627525D LN',
 'A2A IM',
 'AA/ LN',
 'AAL LN',
 'AALB NA',
 'ABBN SW',
 'ABDN LN',
 'ABE SQ',
 'ABF LN',
 'ABI BB',
 'ABN NA',
 'AC FP',
 'ACA FP',
 'ACKB BB',
 'ACS SQ',
 'AD NA',
 'ADEN SW',
 'ADM LN',
 'ADN LN',
 'ADP FP',
 'ADS GY',
 'AENA SQ',
 'AF FP',
 'AGK LN',
 'AGN NA',
 'AGS BB',
 'AHT LN',
 'AI FP',
 'AIR FP',
 'AKE FP',
 'AKZA NA',
 'ALFA SS',
 'ALO FP',
 'ALSYDB DC',
 'ALV GY',
 'AM FP',
 'AMEAS FH',
 'AMFW LN',
 'AMS SE',
 'AMS SQ',
 'ANDR AV',
 'ANTO LN',
 'ARLN GY',
 'ARYN SW',
 'ASM NA',
 'ASML NA',
 'ASSAB SS',
 'ATC NA',
 'ATCOA SS',
 'ATK LN',
 'ATL IM',
 'ATLN SW',
 'ATO FP',
 'AUTO LN',
 'AV/ LN',
 'AVOL SW',
 'AZM IM',
 'AZN LN',
 'BA/ LN',
 'BAB LN',
 'BAER SW',
 'BALDB SS',
 'BALN SW',
 'BARC LN',
 'BARN SE',
 'BAS GY',
 'BATS LN',
 'BAYN GY',
 'BB FP',
 'BBVA SQ',
 'BBY LN',
 'BEI GY',
 'BEZ LN',
 'BILL SS',
 'BION SE',
 'BIRG ID',
 'BKG LN',
 'BKIA SQ',
 'BKT SQ',
 'BL

In [27]:
# stockage
# Export
import pandas as pd
exporter = ExportData(folder_type="verification")
ticker_list_historical_index_memb_unique_dataframe = pd.DataFrame(ticker_list_historical_index_memb_unique)
exporter.export_df(ticker_list_historical_index_memb_unique_dataframe, "ticker_list_historical_index_memb_unique")

WindowsPath('C:/Users/lenar25/Documents/Backtesting_M2272/data/verification/ticker_list_historical_index_memb_unique_20260226_192753.xlsx')

In [ ]:
# reformatage des tickers pour coller avec le format ticker souhiaté par bloomberg
tickers_equity = [t.strip() if "Equity" in t else t.strip() + " Equity" for t in ticker_list_historical_index_memb_unique]

tickers_equity

['1844030D NA Equity',
 '1986986D LN Equity',
 '1CNHI IM Equity',
 '1COV GY Equity',
 '2462936D LN Equity',
 '2578464D GY Equity',
 '2627525D LN Equity',
 'A2A IM Equity',
 'AA/ LN Equity',
 'AAL LN Equity',
 'AALB NA Equity',
 'ABBN SW Equity',
 'ABDN LN Equity',
 'ABE SQ Equity',
 'ABF LN Equity',
 'ABI BB Equity',
 'ABN NA Equity',
 'AC FP Equity',
 'ACA FP Equity',
 'ACKB BB Equity',
 'ACS SQ Equity',
 'AD NA Equity',
 'ADEN SW Equity',
 'ADM LN Equity',
 'ADN LN Equity',
 'ADP FP Equity',
 'ADS GY Equity',
 'AENA SQ Equity',
 'AF FP Equity',
 'AGK LN Equity',
 'AGN NA Equity',
 'AGS BB Equity',
 'AHT LN Equity',
 'AI FP Equity',
 'AIR FP Equity',
 'AKE FP Equity',
 'AKZA NA Equity',
 'ALFA SS Equity',
 'ALO FP Equity',
 'ALSYDB DC Equity',
 'ALV GY Equity',
 'AM FP Equity',
 'AMEAS FH Equity',
 'AMFW LN Equity',
 'AMS SE Equity',
 'AMS SQ Equity',
 'ANDR AV Equity',
 'ANTO LN Equity',
 'ARLN GY Equity',
 'ARYN SW Equity',
 'ASM NA Equity',
 'ASML NA Equity',
 'ASSAB SS Equity',
 '

Requete informations sur tickers


## 4. Requete des informations sur l'historique des composants de l'Indice Univers <a id='1'></a>

In [ ]:
from request.request_informations_members import BloombergTickerInfo 

# téléchargement
info = BloombergTickerInfo(verbose=True)

df_info = info.fetch(tickers_equity)

# sauvegarde
info.save_parquet(df_info)

Parquet sauvegardé : C:\Users\lenar25\Documents\Backtesting_M2272\data\initial_data\informations_hist_memb_index.parquet


WindowsPath('C:/Users/lenar25/Documents/Backtesting_M2272/data/initial_data/informations_hist_memb_index.parquet')

In [ ]:
# Visualisation
stockage = ParquetStockage()

informations_hist_memb_index = stockage.charger("informations_hist_memb_index")

informations_hist_memb_index.tail()

,Ticker,Name,Currency,Sector,Industry,Country
975,ABVX FP Equity,ABIVAX SA,EUR,Soins de santé,Biotechnologie,FRANCE
976,AMV0 GY Equity,AUMOVIO SE,EUR,Consommation discrétionnaire,Composants automobiles,GERMANY
977,CAMX SS Equity,CAMURUS AB,SEK,Soins de santé,Produits pharmaceutiques,SWEDEN
978,FTK GY Equity,FLATEXDEGIRO SE,EUR,Finance,Marchés de capitaux,GERMANY
979,NDX1 GY Equity,NORDEX SE,EUR,Industrie,Matériel électrique,GERMANY


In [ ]:
# exporter pour vérification en excel
exporter = ExportData(folder_type="verification")
exporter.export_df(informations_hist_memb_index, "informations_hist_memb_index")

WindowsPath('C:/Users/lenar25/Documents/Backtesting_M2272/data/verification/informations_hist_memb_index_20260226_194944.xlsx')

 Requeter l'historique


## 5. Requete des historique des prix des composants de l'Indice Univers <a id='1'></a>

In [ ]:
from request.request_historical_data import BloombergPXLastHistory

# fonction de téléchargement des prix de clôture 
px = BloombergPXLastHistory(verbose=True)


df_prices = px.fetch(tickers=tickers_equity, start="2015-09-01", end="2026-02-20", frequency="DAILY")

df_prices.tail()

# sauvegarde dans un fichier de stockage
px.save_parquet(df_prices)

OK: 1844030D NA Equity -> 979 points
OK: 1986986D LN Equity -> 1623 points
OK: 1CNHI IM Equity -> 2661 points
OK: 1COV GY Equity -> 2594 points
OK: 2462936D LN Equity -> 2253 points
OK: 2578464D GY Equity -> 2279 points
OK: 2627525D LN Equity -> 1004 points
OK: A2A IM Equity -> 2661 points
OK: AA/ LN Equity -> 1398 points
OK: AAL LN Equity -> 2648 points
OK: AALB NA Equity -> 2683 points
OK: ABBN SW Equity -> 2633 points
OK: ABDN LN Equity -> 2648 points
OK: ABE SQ Equity -> 750 points
OK: ABF LN Equity -> 2648 points
OK: ABI BB Equity -> 2683 points
OK: ABN NA Equity -> 2626 points
OK: AC FP Equity -> 2683 points
OK: ACA FP Equity -> 2683 points
OK: ACKB BB Equity -> 2683 points
OK: ACS SQ Equity -> 2681 points
OK: AD NA Equity -> 2683 points
OK: ADEN SW Equity -> 2633 points
OK: ADM LN Equity -> 2648 points
OK: ADN LN Equity -> 494 points
OK: ADP FP Equity -> 2683 points
OK: ADS GY Equity -> 2658 points
OK: AENA SQ Equity -> 2681 points
OK: AF FP Equity -> 2683 points
OK: AGK LN Equi

WindowsPath('C:/Users/lenar25/Documents/Backtesting_M2272/data/initial_data/historical_price_memb_index.parquet')

In [ ]:
# visualsiation
stockage = ParquetStockage()

historical_price_memb_index = stockage.charger("historical_price_memb_index")

historical_price_memb_index.tail()

,Date,Ticker,Price
2281884,2026-02-16,VSURE SS Equity,9.899
2281885,2026-02-17,VSURE SS Equity,10.000
2281886,2026-02-18,VSURE SS Equity,10.014
2281887,2026-02-19,VSURE SS Equity,9.804
2281888,2026-02-20,VSURE SS Equity,9.752


Data Management


## 6. Data MAnagement des prix historiques <a id='1'></a>

**Objectif** : obtenir quelques chose de compatibles pour backtester

In [8]:
from request.utils import DataManagement  

dm = DataManagement()

df_wide = dm.format_historical_price(historical_price_memb_index)

df_wide.tail()

Ticker,Date,1844030D NA Equity,1986986D LN Equity,1CNHI IM Equity,1COV GY Equity,1U1 GY Equity,2261689D LN Equity,2462936D LN Equity,2578464D GY Equity,2627525D LN Equity,...,WTB LN Equity,YAR NO Equity,ZAL GY Equity,ZC FP Equity,ZEAL DC Equity,ZEG LN Equity,ZO1 GY Equity,ZOT SQ Equity,ZURN SE Equity,ZURN SW Equity
2685,2026-02-16,NaN,NaN,10.52,NaN,24.60,NaN,NaN,NaN,NaN,...,2691.0,446.0,21.14,NaN,390.1,1640.0,NaN,NaN,557.6,557.6
2686,2026-02-17,NaN,NaN,11.02,NaN,24.45,NaN,NaN,NaN,NaN,...,2738.0,445.0,21.56,NaN,388.0,1695.0,NaN,NaN,564.0,564.0
2687,2026-02-18,NaN,NaN,10.64,NaN,24.15,NaN,NaN,NaN,NaN,...,2757.0,454.3,21.16,NaN,396.4,1785.0,NaN,NaN,568.0,568.0
2688,2026-02-19,NaN,NaN,11.06,NaN,23.55,NaN,NaN,NaN,NaN,...,2737.0,460.0,21.03,NaN,382.8,1790.0,NaN,NaN,558.8,558.8
2689,2026-02-20,NaN,NaN,10.94,NaN,22.55,NaN,NaN,NaN,NaN,...,2730.0,467.8,20.69,NaN,392.9,1815.0,NaN,NaN,567.8,567.8


verification


## 7. Vérification finale <a id='1'></a>


In [ ]:
# benchmark / taux sans risque pour portefeuille long/short
estron_index_px_last = stockage.charger("ESTRON Index_PX_LAST")
estron_index_px_last.tail()

,Date,ESTRON Index
1633,2026-02-16,1.929
1634,2026-02-17,1.931
1635,2026-02-18,1.931
1636,2026-02-19,1.933
1637,2026-02-20,1.932


In [ ]:
# Benchmark indice univers
stockage = ParquetStockage()

sxxp_index_px_last = stockage.charger("SXXP Index_PX_LAST")

sxxp_index_px_last.tail()

,Date,SXXP Index
2427,2026-02-16,618.52
2428,2026-02-17,621.29
2429,2026-02-18,628.69
2430,2026-02-19,625.33
2431,2026-02-20,630.56


In [ ]:
# Composants membre indice univers
stockage = ParquetStockage()

index_memb_20260220 = stockage.charger("index_memb_20260220")

index_memb_20260220.tail()

,Date,Ticker
595,1970-01-01,YAR NO
596,1970-01-01,ZAL GY
597,1970-01-01,ZEAL DC
598,1970-01-01,ZEG LN
599,1970-01-01,ZURN SE


In [ ]:
# membres historiques de l'indice univers
stockage = ParquetStockage()

historical_index_memb = stockage.charger("historical_index_memb")

historical_index_memb.tail()

,Date,Ticker
68397,1970-01-01,YAR NO
68398,1970-01-01,ZAL GY
68399,1970-01-01,ZEAL DC
68400,1970-01-01,ZEG LN
68401,1970-01-01,ZURN SE


In [ ]:
# Historiques des informations sur les membres historiques
stockage = ParquetStockage()

informations_hist_memb_index = stockage.charger("informations_hist_memb_index")

informations_hist_memb_index.tail()

,Ticker,Name,Currency,Sector,Industry,Country
975,ABVX FP Equity,ABIVAX SA,EUR,Soins de santé,Biotechnologie,FRANCE
976,AMV0 GY Equity,AUMOVIO SE,EUR,Consommation discrétionnaire,Composants automobiles,GERMANY
977,CAMX SS Equity,CAMURUS AB,SEK,Soins de santé,Produits pharmaceutiques,SWEDEN
978,FTK GY Equity,FLATEXDEGIRO SE,EUR,Finance,Marchés de capitaux,GERMANY
979,NDX1 GY Equity,NORDEX SE,EUR,Industrie,Matériel électrique,GERMANY


In [ ]:
# Historique de prix des membres de l'indice univers
stockage = ParquetStockage()

historical_price_memb_index = stockage.charger("historical_price_memb_index")

historical_price_memb_index.tail()

,Date,Ticker,Price
2281884,2026-02-16,VSURE SS Equity,9.899
2281885,2026-02-17,VSURE SS Equity,10.000
2281886,2026-02-18,VSURE SS Equity,10.014
2281887,2026-02-19,VSURE SS Equity,9.804
2281888,2026-02-20,VSURE SS Equity,9.752


In [17]:
from request.utils import DataManagement  

dm = DataManagement()

df_wide = dm.format_historical_price(historical_price_memb_index)

df_wide.tail()

Ticker,Date,1844030D NA Equity,1986986D LN Equity,1CNHI IM Equity,1COV GY Equity,1U1 GY Equity,2261689D LN Equity,2462936D LN Equity,2578464D GY Equity,2627525D LN Equity,...,WTB LN Equity,YAR NO Equity,ZAL GY Equity,ZC FP Equity,ZEAL DC Equity,ZEG LN Equity,ZO1 GY Equity,ZOT SQ Equity,ZURN SE Equity,ZURN SW Equity
2685,2026-02-16,NaN,NaN,10.52,NaN,24.60,NaN,NaN,NaN,NaN,...,2691.0,446.0,21.14,NaN,390.1,1640.0,NaN,NaN,557.6,557.6
2686,2026-02-17,NaN,NaN,11.02,NaN,24.45,NaN,NaN,NaN,NaN,...,2738.0,445.0,21.56,NaN,388.0,1695.0,NaN,NaN,564.0,564.0
2687,2026-02-18,NaN,NaN,10.64,NaN,24.15,NaN,NaN,NaN,NaN,...,2757.0,454.3,21.16,NaN,396.4,1785.0,NaN,NaN,568.0,568.0
2688,2026-02-19,NaN,NaN,11.06,NaN,23.55,NaN,NaN,NaN,NaN,...,2737.0,460.0,21.03,NaN,382.8,1790.0,NaN,NaN,558.8,558.8
2689,2026-02-20,NaN,NaN,10.94,NaN,22.55,NaN,NaN,NaN,NaN,...,2730.0,467.8,20.69,NaN,392.9,1815.0,NaN,NaN,567.8,567.8
